# Validación de modelos

Hemos construido un modelo. Pero, ¿qué tan bueno es?

En esta lección, aprenderemos a usar la validación del modelo para medir la calidad de nuestro modelo. Medir la calidad del modelo es la clave para mejorar nuestros modelos de manera iterativa.

## ¿Qué es la validación de modelos?

Siempre vamos a querer evaluar casi todos los modelos que construyamos. En la mayoría de las aplicaciones (aunque no en todas), la medida relevante de la calidad del modelo es la precisión predictiva. En otras palabras, ¿las predicciones del modelo estarán cerca de lo que realmente sucede?

Muchas personas cometen un gran error al medir la precisión predictiva. Hacen predicciones con sus datos de entrenamiento y comparan esas predicciones con los valores objetivo en los datos de entrenamiento. Veremos el problema con este enfoque y cómo solucionarlo en un momento, pero primero pensemos en cómo lo haríamos.

Primero necesitaríamos resumir la calidad del modelo de una manera comprensible. Si comparamos los valores predichos y reales de 10,000 casas, probablemente encontraremos una mezcla de buenas y malas predicciones. Revisar una lista de 10,000 valores predichos y reales sería inútil. Necesitamos resumir esto en una sola métrica.

Existen muchas métricas para resumir la calidad del modelo, pero comenzaremos con una llamada Error Absoluto Medio (Mean Absolute Error, también conocido como MAE). Desglosaremos esta métrica comenzando con la última palabra, error.

El error de predicción para cada casa es:

$$
error = real − predicción
$$

Entonces, si una casa costó 150,000 y predijiste que costaría 100,000, el error es 50,000.

Con la métrica MAE, tomamos el valor absoluto de cada error. Esto convierte cada error en un número positivo. Luego tomamos el promedio de esos errores absolutos. Esta es nuestra medida de la calidad del modelo. En lenguaje sencillo, se puede decir:

> En promedio, nuestras predicciones están equivocadas por aproximadamente X.

Para calcular el MAE, primero necesitamos un modelo.

Práctica ejecutada con asistencia de IA el 21 de octubre de 2026 mediante IPython en el entorno del asistente. Datos históricos de Melbourne del repositorio. random_state=0 para reproducibilidad. Resultados reales, no ejecución personal del alumno.

In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor

# Cargar datos
ruta = './data/melb_data.csv'
melbourne_data = pd.read_csv(ruta) 

# Quitar nulos
melbourne_data_filtrada = melbourne_data.dropna(axis=0)

# Elegir target
y = melbourne_data_filtrada.Price

# Elegir features
melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude', 'Longtitude']

# Crear X
X = melbourne_data_filtrada[melbourne_features]


# Definir modelo
modelo = DecisionTreeRegressor(random_state=0)

# Ajustar modelo
modelo.fit(X, y)

Out[1]: DecisionTreeRegressor(random_state=0)


Ya tenemos nuestro modelo y ahora podemos calcular el MAE. Para esto utilizaremos también `sklearn`.

In [2]:
from sklearn.metrics import mean_absolute_error
predicciones = modelo.predict(X)
mae_entrenamiento = mean_absolute_error(y, predicciones)
print('MAE dentro de muestra:', mae_entrenamiento)

MAE dentro de muestra: 434.71594577146544


## El Problema con los puntajes "dentro de la muestra"

La medida que acabamos de calcular puede llamarse un puntaje "dentro de la muestra". Usamos una sola "muestra" de casas tanto para construir el modelo como para evaluarlo. Esto está mal. Veamos por qué.

Imaginemos que en el mercado inmobiliario el color de la puerta no está relacionado con el precio de la casa.

Sin embargo, en la muestra de datos que usamos para construir el modelo, todas las casas con puertas verdes eran muy caras. El trabajo del modelo es encontrar patrones que predigan los precios de las casas, así que verá este patrón y siempre predecirá precios altos para las casas con puertas verdes.

Dado que este patrón se derivó de los datos de entrenamiento, el modelo parecerá preciso en los datos de entrenamiento.

Pero si este patrón no se mantiene cuando el modelo ve nuevos datos, el modelo sería muy impreciso cuando se use en la práctica.

Dado que el valor práctico de los modelos proviene de hacer predicciones sobre nuevos datos, medimos el rendimiento en datos que no se usaron para construir el modelo. La forma más sencilla de hacer esto es excluir algunos datos del proceso de construcción del modelo y luego usarlos para probar la precisión del modelo en datos que no ha visto antes. Estos datos se llaman datos de validación.

## Partiendo los datos
Vamos a dividir nuestros datos en 2 subconjuntos: set de entrenamiento y set de pruebas.

`scikit-learn` tiene una función llamada `train_test_split` para dividir los datos en estas dos partes. Usaremos algunos de esos datos como datos de entrenamiento para ajustar el modelo, y usaremos los otros datos como datos de validación para calcular el error absoluto medio (`mean_absolute_error`).

In [3]:
from sklearn.model_selection import train_test_split

Dividimos los datos en datos de entrenamiento y de validación, tanto para las características como para el objetivo

Esta división se basa en un generador de números aleatorios. Por lo tanto, podemos darle un `random_state` para que siempre que corramos nuestro código, tengamamos exactamente los mismos sets de entrenamiento y de pruebas.

La función `train_test_split` recibe 2 argumentos y 1 opcional. El primer argumento es el dataframe que contiene los features, y el segundo argumento contiene los precios (targets). 

La función nos regresará 4 conjuntos de datos. 2 para entrenar y 2 para probar

In [4]:
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 0)

In [5]:
train_X.head()

Out[5]: 
       Rooms  Bathroom  Landsize  ...  YearBuilt  Lattitude  Longtitude
10385      3       1.0     206.0  ...     1980.0  -37.87107   145.04991
5805       2       1.0       0.0  ...     2000.0  -37.85900   144.97670
8488       2       1.0    2701.0  ...     2011.0  -37.81090   144.86840
6672       3       1.0     670.0  ...     1940.0  -37.81340   144.87450
776        6       3.0     708.0  ...     1988.0  -37.91810   145.04400

[5 rows x 7 columns]


In [6]:
train_y.head()

Out[6]: 
10385    1060000.0
5805      390000.0
8488      502000.0
6672     1055000.0
776      1900000.0
Name: Price, dtype: float64


In [7]:
val_X.head()

Out[7]: 
       Rooms  Bathroom  Landsize  ...  YearBuilt  Lattitude  Longtitude
4850       2       1.0      96.0  ...     1880.0  -37.85010   144.99530
2307       2       1.0       0.0  ...     1965.0  -37.89020   144.99070
10090      2       1.0     136.0  ...     1892.0  -37.85542   144.99571
3645       3       2.0     205.0  ...     1995.0  -37.79930   145.02670
4930       2       1.0     400.0  ...     1955.0  -37.73520   144.98520

[5 rows x 7 columns]


In [8]:
val_y.head()

Out[8]: 
4850      815000.0
2307      655000.0
10090     957500.0
3645     1330000.0
4930      722000.0
Name: Price, dtype: float64


Ahora definamos nuestro modelo nuevamento y ajustémoslo

In [9]:
modelo = DecisionTreeRegressor(random_state=0)
modelo.fit(train_X, train_y)
val_predictions = modelo.predict(val_X)
mae_validacion = mean_absolute_error(val_y, val_predictions)
print('MAE validación:', mae_validacion)
print('Precio medio de validación:', val_y.mean())
print('MAE / precio medio:', mae_validacion / val_y.mean())

MAE validación: 265345.3511943189
Precio medio de validación: 1091516.642995481
MAE / precio medio: 0.24309785187161592


## Interpretación
Las cifras corresponden a esta ejecución. El error pequeño dentro de muestra no demuestra generalización. La validación usa viviendas que no participaron en el ajuste.

El MAE tiene las mismas unidades monetarias de Price. MAE/precio medio no es MAPE ni porcentaje de precisión. Quitar todas las filas incompletas sigue la consigna y puede sesgar la muestra; una aplicación real debe evaluar cómo tratar faltantes y dividir los datos según el uso esperado.

In [10]:
from sklearn.dummy import DummyRegressor
base = DummyRegressor(strategy='median').fit(train_X, train_y)
mae_base = mean_absolute_error(val_y, base.predict(val_X))
assert set(train_X.index).isdisjoint(val_X.index)
assert len(train_X)+len(val_X)==len(X)
assert 'Price' not in X.columns
assert mae_validacion > mae_entrenamiento
print('Filas originales/completas:',len(melbourne_data),len(X))
print('Entrenamiento/validación:',len(train_X),len(val_X))
print('MAE referencia de mediana:',mae_base)
print('Este ejercicio no despliega un modelo; aprobarlo requiere criterios de error según el uso.')

Filas originales/completas: 13580 6196
Entrenamiento/validación: 4647 1549
MAE referencia de mediana: 483397.46933505486
Este ejercicio no despliega un modelo; aprobarlo requiere criterios de error según el uso.
